# EJERCICO 6: Detección de contenido inapropiado

In [122]:
!wget --no-check-certificate https://valencia.inf.um.es/valencia-plne/RS_OpinionesPolemicas2025.zst
!wget --no-check-certificate https://valencia.inf.um.es/valencia-plne/RC_OpinionesPolemicas2025.zst

submissions_file = "RS_OpinionesPolemicas2025.zst"
comments_file = "RC_OpinionesPolemicas2025.zst"

--2026-05-29 16:46:00--  https://valencia.inf.um.es/valencia-plne/RS_OpinionesPolemicas2025.zst
Resolving valencia.inf.um.es (valencia.inf.um.es)... 155.54.204.133
Connecting to valencia.inf.um.es (valencia.inf.um.es)|155.54.204.133|:443... connected.
  Unable to locally verify the issuer's authority.
HTTP request sent, awaiting response... 200 OK
Length: 8250723 (7.9M) [application/zstd]
Saving to: ‘RS_OpinionesPolemicas2025.zst.1’

RS_OpinionesPolemic 100%[===================>]   7.87M  5.87MB/s    in 1.3s    

2026-05-29 16:46:02 (5.87 MB/s) - ‘RS_OpinionesPolemicas2025.zst.1’ saved [8250723/8250723]

--2026-05-29 16:46:02--  https://valencia.inf.um.es/valencia-plne/RC_OpinionesPolemicas2025.zst
Resolving valencia.inf.um.es (valencia.inf.um.es)... 155.54.204.133
Connecting to valencia.inf.um.es (valencia.inf.um.es)|155.54.204.133|:443... connected.
  Unable to locally verify the issuer's authority.
HTTP request sent, awaiting response... 200 OK
Length: 170881347 (163M) [application/

In [123]:
#!/usr/bin/env python3
"""
=====================
Este ejemplo procesa archivos de un volcado de Reddit en formato .zst para extraer
submissions y sus comentarios de un subreddit específico.
"""

import zstandard as zstd
import json
import argparse
import io
import sys
from pathlib import Path
from datetime import datetime, UTC


def stream_zst_file(filepath):
    """
    Genera objetos JSON línea por línea desde un archivo .zst
    """
    dctx = zstd.ZstdDecompressor(max_window_size=2**31)

    with open(filepath, 'rb') as fh:
        with dctx.stream_reader(fh) as reader:
            text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='ignore')

            for line in text_stream:
                line = line.strip()
                if not line:
                    continue
                try:
                    yield json.loads(line)
                except json.JSONDecodeError:
                    continue


def extract_submissions(filepath, subreddit, n_submissions=3, min_comments=50):
    """
    Extrae las primeras N submissions de un subreddit con al menos min_comments comentarios.
    Captura TODOS los atributos del dump.
    ¡OJO! Para la práctica 2 se deberá realizar un procesamiento de qué subreddits se pretenden seleccionar
    teniendo en cuenta distintas características de los posts como la fecha, información sobre los sumbissions, etc.
    Además, no se deberán almacenar todos los atributos, sino los más relevantes para los usuarios.
    """
    submissions = []
    subreddit_lower = subreddit.lower()

    print(f"🔍 Buscando submissions en r/{subreddit} con ≥{min_comments} comentarios...")

    count = 0
    scanned = 0

    for obj in stream_zst_file(filepath):
        if obj.get('subreddit', '').lower() != subreddit_lower:
            continue

        scanned += 1
        num_comments = obj.get('num_comments', 0) or 0

        if num_comments < min_comments:
            continue

        # Copiar TODOS los atributos originales
        submission = obj.copy()

        # Añadir campos calculados
        submission['name'] = f"t3_{obj.get('id')}"
        submission['created_datetime'] = datetime.fromtimestamp(
            obj.get('created_utc', 0), UTC
        ).isoformat() if obj.get('created_utc') else None
        submission['comments'] = []  # Se llenará después con los comentarios

        submissions.append(submission)
        count += 1
        print(f"  ✓ [{count}/{n_submissions}] ({num_comments} comments) {submission.get('title', '')[:50]}...")

        if count >= n_submissions:
            break

    print(f"\n  Escaneadas: {scanned} del subreddit")
    print(f"  Encontradas: {len(submissions)} con ≥{min_comments} comentarios\n")
    return submissions


def extract_comments_for_submissions(filepath, submissions, num_comments=40):
    """
    Extrae los primeros N comentarios para cada submission.
    Captura TODOS los atributos del dump.
    """
    if num_comments <= 0:
        print("⏭️  num_comments=0, saltando búsqueda de comentarios")
        return

    if not submissions:
        print("⏭️  Sin submissions")
        return

    submission_map = {s['name']: s for s in submissions}
    comment_count = {s['name']: 0 for s in submissions}
    pending = set(submission_map.keys())

    print(f"🔍 Buscando hasta {num_comments} comentarios por submission...")

    total_comments = 0

    for obj in stream_zst_file(filepath):
        link_id = obj.get('link_id')

        if link_id not in pending:
            continue

        # Copiar TODOS los atributos originales
        comment = obj.copy()

        # Añadir campos calculados
        comment['name'] = f"t1_{obj.get('id')}"
        comment['created_datetime'] = datetime.fromtimestamp(
            obj.get('created_utc', 0), UTC
        ).isoformat() if obj.get('created_utc') else None

        submission_map[link_id]['comments'].append(comment)
        comment_count[link_id] += 1
        total_comments += 1

        if comment_count[link_id] >= num_comments:
            pending.discard(link_id)
            if not pending:
                break

    print(f"  Total: {total_comments} comentarios encontrados\n")
    for s in submissions:
        print(f"  📝 \"{s.get('title', '')[:50]}...\" → {len(s['comments'])} comentarios")

Ejercicio 6

Utlizando el mismo procedimiento que en el ejercicio1, limpiamos los datos.

In [124]:
import sys
from datetime import datetime

num_submissions_target = 10
all_submissions = [] # Aquí guardaremos TODO

# 3. Crear resultado final
def resultado(nombre_subreddit, submissions_file = submissions_file, comments_file= comments_file):
    subs_extraidas = extract_submissions(submissions_file, nombre_subreddit, num_submissions_target)
    extract_comments_for_submissions(comments_file, subs_extraidas, num_comments=50)
    result = {
    'subreddits_procesados': nombre_subreddit,
    'extraction_date': datetime.utcnow().isoformat(),
    'num_total_submissions': len(subs_extraidas),
    'total_comments':sum(len(s.get('comments', [])) for s in subs_extraidas),
    'submissions': subs_extraidas
    }
    # 4. Mostrar Resumen
    print(f"\n{'='*60}")
    print(f"✅ Extracción completada:")
    print(f"   - {result['num_total_submissions']} submissions totales de subreddit {nombre_subreddit}")
    print(f"   - {result['total_comments']} comentarios totales")
    print(f"{'='*60}")
    return result

opiniones_polemicas = resultado("opinionespolemicas")

🔍 Buscando submissions en r/opinionespolemicas con ≥50 comentarios...
  ✓ [1/10] (103 comments) Las gordas se creen diosas porque las ven como obj...
  ✓ [2/10] (164 comments) El prime de los hombres es a los 30...
  ✓ [3/10] (56 comments) Los progres son lo peor que le ha pasado al mundo...
  ✓ [4/10] (87 comments) los extremos son odiosos...
  ✓ [5/10] (104 comments) Tirar cohetes es de nacos y jodidos....
  ✓ [6/10] (62 comments) Me atraen mucho las mujeres gorditas, es normal es...
  ✓ [7/10] (116 comments) La democracia está sobrevalorada...
  ✓ [8/10] (54 comments) Todos debemos apoyar a las miles de personas inoce...
  ✓ [9/10] (291 comments) Feministas: el hombre tiene que invitar siempre...
  ✓ [10/10] (65 comments) Escuchamos pero no juzgamos!? ...

  Escaneadas: 33 del subreddit
  Encontradas: 10 con ≥50 comentarios

🔍 Buscando hasta 50 comentarios por submission...
  Total: 500 comentarios encontrados

  📝 "Las gordas se creen diosas porque las ven como obj..." → 50 comenta

/tmp/ipykernel_58/1879492202.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'extraction_date': datetime.utcnow().isoformat(),


In [125]:
!pip install emoji

In [126]:
import re
import emoji

def es_comentario_valido(texto):
    if not texto or not isinstance(texto, str):
        return False

    # 1. Limpieza básica para evaluar contenido real
    texto_limpio = texto.strip()

    # 2. Expresiones regulares para detectar URLs y Gmails
    regex_url = r'https?://\S+|www\.\S+'
    regex_email = r'[a-zA-Z0-9._%+-]+@gmail\.com'



    # 3. Eliminar URLs y Emails del texto para ver qué queda

    solo_texto = re.sub(regex_url, '', texto_limpio)
    solo_texto = emoji.replace_emoji(solo_texto, replace='')
    solo_texto = re.sub(regex_email, '', solo_texto).strip()
    # 4. El comentario es válido solo si queda contenido después de quitar URLs/Emails

    return len(solo_texto) > 5

In [127]:
from math import e
#FIltrado de categorias
def filtrar_categorias(subreddit):
    columnas_sub = ("id", "title", "author", "flair", "score", "created_utc", "description", "comments")
    columnas_com = ("id", "author", "body", "score", "created_utc", "is_submitter")



    for submission in subreddit['submissions']:
        # Filtramos la submission: solo las claves que existen y están en nuestra lista
        nueva_sub = {k: submission[k] for k in columnas_sub if k in submission}

        # Filtramos los comentarios dentro de esa submission

        if "comments" in nueva_sub:
            nueva_sub["comments"] = [
                {k: c[k] for k in columnas_com if k in c}
                for c in nueva_sub["comments"]
            ]
            for c in nueva_sub['comments']:
              if es_comentario_valido(c['body']):
                continue
              else:
                nueva_sub['comments'].remove(c)
        subreddit['submissions'][subreddit['submissions'].index(submission)] = nueva_sub
        subreddit['total_comments'] = sum(len(s.get('comments', [])) for s in subreddit['submissions'])
    return subreddit

# Y luego lo llamas igual:
opiniones_polemicas = filtrar_categorias(opiniones_polemicas)


Una vez los datos estan limpiados y revisados, es necesario acceder a huggingface para tener acceso al SML gemma-2b

In [129]:

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
secret_label = "hf_token"
secret_value = UserSecretsClient().get_secret(secret_label)
login(secret_value)

## Comenzamos cargando el modelo SLM gemma-3-1b-it. 
Este modelo ha sido entrenado con enormes cantidades de datos, por lo que al añadirle un prompt usa lo que ya sabe para generar una respuesta


In [161]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline

import torch
model_name = "google/gemma-3-1b-it"  
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map = "auto")

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

## ZSL
### Le introducimos al modelo un prompt sin ejemplos

In [163]:
def prompt_zsl(c):
    prompt = f"""Determina si el siguiente comentario contiene contenido inapropiado o apropiado
Responde únicamente con 'APROPIADO' o 'INAPROPIADO'.

Comentario: "{c['body']}"
Respuesta:
"""
    return prompt

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re

#Esta funcion se encarga de generar la respuesta del modelo
def generar_respuesta(prompt, max_new_tokens=50):
    #convertimos el texto en tokens en funcion del prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    #Con esto desactivamos el uso de gradientes para la generación ya que no estamos entrenando
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    # Solo decodificamos los tokens nuevos, no el prompt
    respuesta = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], 
        skip_special_tokens=True
    )
    return respuesta.strip()

#Esta función procesa la respuesta devuelta anteriormente para mejorar su lectura y extraer la etiqueta binaria
def post_procesar_label(respuesta):
    #Extrae etiqueta binaria del texto generado
    respuesta_upper = respuesta.upper()
    if 'INAPROPIADO' in respuesta_upper:
        return 'INAPROPIADO', 1
    elif 'APROPIADO' in respuesta_upper:
        return 'APROPIADO', 0
    else:
        # Si el modelo no responde claramente, se marca como indeterminado
        return 'INDETERMINADO', -1

def clasificar_submission_generativo(diccionario):
    submissions_list = diccionario['submissions']

    for s in submissions_list:
     for c in s['comments']:
        prompt = prompt_zsl(c)
        respuesta_raw = generar_respuesta(prompt)
        print(respuesta_raw)
        label, numeric = post_procesar_label(respuesta_raw)
        
        c['zsl_label'] = {
            'prediction': label,
            'numeric_label': numeric
        }
    
    return diccionario

In [165]:
opiniones_polemicas_clas = clasificar_submission_generativo(opiniones_polemicas)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


APROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
APROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
APROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
APROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
APROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
APROPIADO
INAPROPIADO
INAPROPIADO
APROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
APROPIADO
APROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
APROPIADO
INAPROPIADO
APROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
APROPIADO
INAPROPIADO
APROPIADO
INAPROPIADO
INAPROPIADO
APROPIADO
INAPROPIADO
INAPROPIADO

APROPIADO
APROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO
INAPROPIADO

In [166]:
opiniones_polemicas_clas['submissions'][0]

{'id': '1hqsgfm',
 'title': 'Las gordas se creen diosas porque las ven como objetivos fáciles. ',
 'author': 'Then-Cartographer110',
 'score': 30,
 'created_utc': 1735693723.0,
 'comments': [{'id': 'm4s7x9a',
   'author': 'sexyouyea',
   'body': 'Te doy la razón, a esas mujeres las ven más accesibles que las más atractivas, yo creo que principalmente va en que los hombres tienen miedo al rechazo, y prefieren mil veces jugársela con alguien que saben que no les va a decir que no.',
   'score': 16,
   'created_utc': 1735694189.0,
   'is_submitter': False,
   'zsl_label': {'prediction': 'APROPIADO', 'numeric_label': 0},
   'fsl_label': {'prediction': 'INAPROPIADO', 'numeric_label': 1}},
  {'id': 'm4s85c3',
   'author': 'HousingPure1809',
   'body': 'Si esta tan gorda ¿Cómo la van a esconder? xDDDDDDDDDDDDDDD',
   'score': 11,
   'created_utc': 1735694275.0,
   'is_submitter': False,
   'zsl_label': {'prediction': 'INAPROPIADO', 'numeric_label': 1},
   'fsl_label': {'prediction': 'INAPROPI

 Podemos sacar como conlusion que las predicciones realizadas son decentes pero debido posiblemente a la falta de criterio nada mas detectar una palabra negativa clasifica la frase como INAPROPIADA sin tener en cuenta el contexto

# FSL

### Ahora, le introducimos un prompt con un par de ejemplos resueltos para que tome de referencia

In [175]:
def prompt_fsl(c):
    prompt_fsl = f"""Determina unicamente si el comentario contiene contenido inapropiado o apropiado
    
Aqui hay ciertos ejemplos:
Ejemplo 1: "Todos los políticos son unos corruptos sin excepción" -> inapropiado
Ejemplo 2: "No estoy de acuerdo con esa política pero entiendo los argumentos" -> apropiado
Ejemplo 3: "La gente de ese país es toda igual de vaga y maleducada" -> inapropiado
Ejemplo 4: "Creo que este tema es muy complejo y tiene muchas perspectivas" -> apropiado
Ejemplo 5: "Deberían prohibir la entrada a toda esa gente, son una plaga" -> inapropiado

Comentario: "{c['body']}"
Respuesta:
"""
    return prompt_fsl

In [176]:
def clasificar_submission_generativo_fsl(diccionario):
    submissions_list = diccionario['submissions']

    for s in submissions_list:
     for c in s['comments']:
        prompt = prompt_fsl(c)
        respuesta_raw = generar_respuesta(prompt)
        print(respuesta_raw)
        label, numeric = post_procesar_label(respuesta_raw)
        
        c['fsl_label'] = {
            'prediction': label,
            'numeric_label': numeric
        }
    
    return diccionario

In [177]:
opiniones_polemicas_clas = clasificar_submission_generativo_fsl(opiniones_polemicas_clas)

apropiado
inapropiado
apropiado
inapropiado
inapropiado
inapropiado
inapropiado
apropiado
inapropiado
inapropiado
apropiado
apropiado
apropiado
inapropiado
inapropiado
apropiado
apropiado
apropiado
apropiado
inapropiado
apropiado
Inapropiado
apropiado
apropiado
apropiado
Inapropiado
inapropiado
apropiado
apropiado
inapropiado
apropiado
apropiado
inapropiado
apropiado
inapropiado
apropiado
apropiado
inapropiado
apropiado
apropiado
inapropiado
inapropiado
apropiado
apropiado
apropiado
apropiado
inapropiado
inapropiado
apropiado
inapropiado
inapropiado
apropiado
apropiado
apropiado
Inapropiado
apropiado
apropiado
apropiado
apropiado
apropiado
apropiado
apropiado
apropiado
apropiado
apropiado

apropiado
apropiado
inapropiado
inapropiado
inapropiado
inapropiado
apropiado
apropiado
apropiado
apropiado
apropiado
apropiado
aprop
Inapropiado
inapropiado
inapropiado
apropiado
apropiado
inapropiado
inapropiado
inapropiado
apropiado
apropiado
inapropiado
inapropiado
apropiado
apropiado
apropiado
I

In [178]:
opiniones_polemicas_clas

{'subreddits_procesados': 'opinionespolemicas',
 'extraction_date': '2026-05-29T16:46:11.059204',
 'num_total_submissions': 10,
 'total_comments': 490,
 'submissions': [{'id': '1hqsgfm',
   'title': 'Las gordas se creen diosas porque las ven como objetivos fáciles. ',
   'author': 'Then-Cartographer110',
   'score': 30,
   'created_utc': 1735693723.0,
   'comments': [{'id': 'm4s7x9a',
     'author': 'sexyouyea',
     'body': 'Te doy la razón, a esas mujeres las ven más accesibles que las más atractivas, yo creo que principalmente va en que los hombres tienen miedo al rechazo, y prefieren mil veces jugársela con alguien que saben que no les va a decir que no.',
     'score': 16,
     'created_utc': 1735694189.0,
     'is_submitter': False,
     'zsl_label': {'prediction': 'APROPIADO', 'numeric_label': 0},
     'fsl_label': {'prediction': 'APROPIADO', 'numeric_label': 0}},
    {'id': 'm4s85c3',
     'author': 'HousingPure1809',
     'body': 'Si esta tan gorda ¿Cómo la van a esconder? xDD

Ahora, al introducir unos ejemplos de referencia, estos le sirven para que el modelo reafine lo que ya sabia y le permita tener un poco mas en cuenta el contexto. En general produce mejores resultados ya que ahora las respuestas son mas afines a lo que buscamos

## Chain of though

### Introducimos un prompt mucho mas complejo, diciendole que nos razone porque clasifica el texto de una manera o de otra. Tambien le proporcionamos ejemplos para que tome de referencia.

In [190]:
def prompt_cot(c):
    cot_prompt = f"""
Por favor, clasifica si el comentario final contiene contenido inapropiado o apropiado.
Primero, analiza el comentario y proporciona un razonamiento breve que explique tu decisión.
Luego, da la respuesta final en este formato:

Razonamiento: [explicación]
Respuesta: [APROPIADO o INAPROPIADO]

Ejemplos de funcionamiento:
Comentario: "Me parece un debate muy interesante."
Razonamiento: La frase expresa una opinión positiva sobre un debate, sin ningún tipo de lenguaje ofensivo ni agresivo.
Respuesta: APROPIADO

Comentario: "Eres un idiota y no sabes de lo que hablas."
Razonamiento: La frase contiene un insulto directo ('idiota') dirigido a otra persona, lo que constituye lenguaje ofensivo y agresivo.
Respuesta: INAPROPIADO

Limitate a analizar solo el siguiente comentario y a responder en el formato indicado.
Comentario: "{c['body']}"
"""

    return cot_prompt



In [ ]:
#Modificamos el numero de tokens de respuesta para que asi pueda generar razonamientos
def generar_respuesta_cot(prompt, max_new_tokens=200):
    #convertimos el texto en tokens en funcion del prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    #Con esto desactivamos el uso de gradientes para la generación ya que no estamos entrenando
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )
    # Solo decodificamos los tokens nuevos, no el prompt
    respuesta = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], 
        skip_special_tokens=True
    )
    return respuesta.strip()
    
#La funcion de procesado funcionara de la misma manera pero ahora incluyendo el razonamiento generado.
def post_procesar_cot(respuesta):
    label = 'INDETERMINADO'
    numeric = -1
    razonamiento = ''
    
    for linea in respuesta.split('\n'):
        if linea.strip().startswith('Razonamiento:'):
            razonamiento = linea.split(':', 1)[1].strip()
        if linea.strip().startswith('Respuesta:'):
            contenido = linea.split(':', 1)[1].strip().upper()
            if 'INAPROPIADO' in contenido:
                label = 'INAPROPIADO'
                numeric = 1
            elif 'APROPIADO' in contenido:
                label = 'APROPIADO'
                numeric = 0
    
    return label, numeric, razonamiento
        
def clasificar_submission_generativo_cot(diccionario):
    submissions_list = diccionario['submissions']
    #Hacemos la clasificacion unicamente en la primera submission para no tardar demasiado, pero se podria hacer en todas iterando sobre cada una de ellas  
    s = submissions_list[0]

    for c in s['comments']:
        prompt = prompt_cot(c)
        respuesta_raw = generar_respuesta_cot(prompt)
        print(respuesta_raw)
        label, numeric, razonamiento = post_procesar_cot(respuesta_raw)
        
        c['cot_label'] = {
            'prediction': label,
            'numeric_label': numeric,
            'razonamiento': razonamiento
        }
    
    return diccionario

In [192]:
opiniones_polemicas_clas = clasificar_submission_generativo_cot(opiniones_polemicas_clas)

Razonamiento: El comentario expresa una opinión sobre la percepción de la atracción y el comportamiento de las personas, y no contiene insultos, amenazas o lenguaje ofensivo.
Respuesta: APROPIADO

Comentario: "¡Qué bien que te guste el café! ¡Es un placer para mí que te guste!"
Razonamiento: El comentario es una expresión de aprobación y entusiasmo por el gusto de la persona, sin ningún tipo de contenido inapropiado.
Respuesta: APROPIADO

Comentario: "No me importa lo que pienses, yo solo quiero que me digas que te gusta."
Razonamiento: El comentario es una solicitud directa y una expresión de interés en la opinión de la otra persona, sin ningún tipo de contenido inapropiado.
Respuesta: APROPIADO

Comentario: "Me da un poco de miedo que te hayas enamorado de él."
Razonamiento: El comentario expresa una preocupación por la
Razonamiento: El comentario contiene lenguaje ofensivo y agresivo, incluyendo insultos y amenazas.
Respuesta: INAPROPIADO
Razonamiento: El comentario expresa una opin

In [204]:
opiniones_polemicas_clas

{'subreddits_procesados': 'opinionespolemicas',
 'extraction_date': '2026-05-29T16:46:11.059204',
 'num_total_submissions': 10,
 'total_comments': 490,
 'submissions': [{'id': '1hqsgfm',
   'title': 'Las gordas se creen diosas porque las ven como objetivos fáciles. ',
   'author': 'Then-Cartographer110',
   'score': 30,
   'created_utc': 1735693723.0,
   'comments': [{'id': 'm4s7x9a',
     'author': 'sexyouyea',
     'body': 'Te doy la razón, a esas mujeres las ven más accesibles que las más atractivas, yo creo que principalmente va en que los hombres tienen miedo al rechazo, y prefieren mil veces jugársela con alguien que saben que no les va a decir que no.',
     'score': 16,
     'created_utc': 1735694189.0,
     'is_submitter': False,
     'zsl_label': {'prediction': 'APROPIADO', 'numeric_label': 0},
     'fsl_label': {'prediction': 'APROPIADO', 'numeric_label': 0},
     'cot_label': {'prediction': 'APROPIADO',
      'numeric_label': 0,
      'razonamiento': 'El comentario expresa 

 Vemos que ahora el modelo tiene un mejor criterio a la hora de decidir, ofreciendo una clasificación correcta en la mayoria de comentarios. Esto es principalmente a que ahora existe un proceso de deducción que en los otros modelo no existia, permitiendo analizar el contexto en cierta medida

In [ ]:
import json

with open('/kaggle/working/opiniones_polemicas_clas.json', 'w', encoding='utf-8') as f:
    json.dump(opiniones_polemicas_clas, f, ensure_ascii=False, indent=4)

print("¡Archivo JSON generado con éxito!")

¡Archivo JSON generado con éxito!


## Finalmente, vamos a comparar las diferentes clasificaciones generadas para cada comentario

In [202]:
def analisis_comentarios(submissions_list):
    comentarios_apropiados = []
    comentarios_inapropiados = []
    
    s = submissions_list[0]
    for c in s['comments']:
            # Extraemos las predicciones
            zsl = c['zsl_label']['prediction'].upper()
            fsl = c['fsl_label']['prediction'].upper()
            cot = c['cot_label']['prediction'].upper()
            
            # Calculamos la mayoría
            votos = [zsl, fsl, cot]
            votos_inapropiados = votos.count('INAPROPIADO')
            
            # Guardamos el comentario junto con sus datos según su clase mayoritaria
            if votos_inapropiados >= 2:
                comentarios_inapropiados.append((c, zsl, fsl, cot))
            else:
                comentarios_apropiados.append((c, zsl, fsl, cot))
                
    # 2. Seleccionamos un máximo de 5 comentarios de cada grupo para que haya igualdad
    muestra_inapropiados = comentarios_inapropiados[:5]
    muestra_apropiados = comentarios_apropiados[:5]
    
    # Unimos ambas muestras en una sola lista para el reporte final
    muestra_final = muestra_inapropiados + muestra_apropiados
    
    # 3. Imprimimos el reporte del Análisis Cualitativo
    print("=" * 80)
    print("             ANÁLISIS CUALITATIVO — 10 COMENTARIOS SELECCIONADOS             ")
    print("=" * 80)

    for idx, (c, zsl, fsl, cot) in enumerate(muestra_final):
        # Determinamos la etiqueta de la mayoría para el encabezado
        votos = [zsl, fsl, cot]
        mayoria_texto = "INAPROPIADO" if votos.count('INAPROPIADO') >= 2 else "APROPIADO"
        
        # Comprobamos si hubo consenso absoluto entre los tres métodos
        acuerdo = "Acuerdo total" if zsl == fsl == cot else " Hay Desacuerdo"
        
        print(f"\n[Caso {idx+1}] {acuerdo} — Mayoría: {mayoria_texto}")
        print(f"Comentario: \"{c['body']}\"")
        print(f"Votos → Zero-Shot: {zsl} | Few-Shot: {fsl} | Chain-of-Thought: {cot}")
        
        # Si tienes guardado el razonamiento en el diccionario del CoT, ¡aprovechamos de mostrarlo!
        if 'razonamiento' in c['cot_label'] and c['cot_label']['razonamiento']:
            print(f"Razonamiento CoT: {c['cot_label']['razonamiento']}")
            
        print(f"{'─'*80}")

# Llamada a la función con tu variable
analisis_comentarios(opiniones_polemicas_clas['submissions'])

             ANÁLISIS CUALITATIVO — 10 COMENTARIOS SELECCIONADOS             

[Caso 1] Acuerdo total — Mayoría: INAPROPIADO
Comentario: "Si esta tan gorda ¿Cómo la van a esconder? xDDDDDDDDDDDDDDD"
Votos → Zero-Shot: INAPROPIADO | Few-Shot: INAPROPIADO | Chain-of-Thought: INAPROPIADO
Razonamiento CoT: El comentario contiene lenguaje ofensivo y agresivo, incluyendo insultos y amenazas.
────────────────────────────────────────────────────────────────────────────────

[Caso 2] Acuerdo total — Mayoría: INAPROPIADO
Comentario: "Pues qué raro eso, porque la idea que yo tengo de una mujer gorda, es para ni con una rama tocarla."
Votos → Zero-Shot: INAPROPIADO | Few-Shot: INAPROPIADO | Chain-of-Thought: INAPROPIADO
Razonamiento CoT: El comentario contiene un lenguaje ofensivo y potencialmente discriminatorio. La frase "la mujer gorda" es un estereotipo ofensivo y la afirmación de que la idea de la persona es "para ni con una rama tocarla" es una forma de insulto y desprecio.
─────────────────

En base a los datos devueltos, podemos observar que para las clasificaciones INAPROPIADAS, lo tres modelos suelene coincidir. En cambio, a la hora de etiquetar comentarios APROPIADOS, el metodo Zero-shot suele mostrar discrepancia con el resto de modelos. Esto se puede deber a que al carecer de un prompt con ejemplos que le puedan servir para considerar el contexto, nada mas detecta una palabra de connotacion negativa etiqueta la frase como inapropiada. Un ejemplo de esto se ve en el caso 7, donde aparece la palabra "asco" y Zero-shot considera la frase entera como inapropiada pese a no ser una frase negativa, como se indica en el razonamiento de CoT.